# YieldGuard — tabular (SECOM) evaluation

Evaluates the anomaly detector on SECOM. **CPU is fine** — the dataset is 1,567 rows by 590
sensors and the whole notebook runs in under a minute. No GPU, no uploads, no credentials:
SECOM comes from the UCI repo over HTTP.

Rather than shipping a pickle up here, the notebook **retrains the locked config**
(`VarianceThreshold(0.05)` + `IsolationForest(n=300, contamination=0.05, max_features=0.7)`,
`random_state=42`). That is deterministic, so it reproduces the checkpoint in
`src/models/tabular/checkpoints/`.

What this adds over the numbers already in `model_meta.json`:

- **bootstrap confidence intervals** — validation has 21 failing lots, so a headline recall
  of 0.286 means *6 of 21*. The error bar on that is the most important number here.
- **baseline comparisons** — unsupervised anomaly detection has to beat guessing to earn
  its place, and on SECOM that is not a given.
- **a supervised reference** — worth knowing how much of the difficulty is the method
  versus the dataset.
- **a threshold sweep** — the shipped 0.373 is one point on a curve the demo should be
  honest about.

In [ ]:
#@title 1. Setup + download SECOM
!pip -q install ucimlrepo

import numpy as np, pandas as pd, warnings, json
warnings.filterwarnings("ignore")
rng = np.random.default_rng(42)

# Constants copied from src/models/tabular/data_prep.py — changing any of these
# changes the split, and the numbers stop being comparable to the repo.
MISSING_DROP_THRESHOLD = 0.80
VAL_FRACTION  = 0.20
RANDOM_STATE  = 42
PASS_LABEL, FAIL_LABEL = -1, 1     # UCI convention

from ucimlrepo import fetch_ucirepo
secom = fetch_ucirepo(id=179)
X_df = secom.data.features.copy()
X_df.columns = [f"sensor_{i}" for i in range(X_df.shape[1])]
y_raw = secom.data.targets.iloc[:, 0].to_numpy()
y_all = (y_raw == FAIL_LABEL).astype(int)          # 1 = fail

print(f"rows {len(X_df)}  sensors {X_df.shape[1]}  fails {y_all.sum()} "
      f"({y_all.mean()*100:.1f}%)  missing {X_df.isna().mean().mean()*100:.1f}%")

In [ ]:
#@title 2. Preprocess — same pipeline as data_prep.py
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

keep = X_df.columns[X_df.isna().mean() <= MISSING_DROP_THRESHOLD]
X_df = X_df[keep]
print(f"kept {len(keep)} sensors after dropping >{MISSING_DROP_THRESHOLD:.0%} missing")

Xtr_df, Xva_df, y_train, y_val = train_test_split(
    X_df, y_all, test_size=VAL_FRACTION, stratify=y_all, random_state=RANDOM_STATE)

# median imputation fit on TRAIN only — fitting on all rows leaks val into train
med = np.nan_to_num(np.nanmedian(Xtr_df.to_numpy(dtype=float), axis=0))
fill = lambda A: np.where(np.isnan(A), med, A)
scaler  = StandardScaler().fit(fill(Xtr_df.to_numpy(dtype=float)))
X_train = scaler.transform(fill(Xtr_df.to_numpy(dtype=float)))
X_val   = scaler.transform(fill(Xva_df.to_numpy(dtype=float)))

print(f"train {X_train.shape}  fails {y_train.sum()}")
print(f"val   {X_val.shape}  fails {y_val.sum()}   <-- every metric below rests on these")

In [ ]:
#@title 3. Train the locked config + score
from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import IsolationForest

vt = VarianceThreshold(0.05).fit(X_train)
Xtr_v, Xva_v = vt.transform(X_train), vt.transform(X_val)
print("features after variance filter:", Xtr_v.shape[1])

# contamination is inert here: it sets the .predict() offset, not score_samples(),
# and every metric below is computed from score_samples. Kept only to match the repo.
iforest = IsolationForest(n_estimators=300, contamination=0.05,
                          max_features=0.7, random_state=RANDOM_STATE).fit(Xtr_v)

# calibration on training PASS rows only, exactly as anomaly.py does it
ss_tr_pass = iforest.score_samples(Xtr_v[y_train == 0])
cal_max, cal_min = ss_tr_pass.max(), ss_tr_pass.min()
to_score = lambda Z: np.clip((cal_max - iforest.score_samples(Z)) / (cal_max - cal_min), 0, 1)

scores = to_score(Xva_v)
print(f"calibration  max={cal_max:.4f}  min={cal_min:.4f}")
print(f"val anomaly scores: min {scores.min():.3f}  median {np.median(scores):.3f}  "
      f"max {scores.max():.3f}")

In [ ]:
#@title 4. Metrics + bootstrap CIs  — the honest headline
from sklearn.metrics import (precision_recall_fscore_support, roc_auc_score,
                             average_precision_score, confusion_matrix)

THRESHOLD = 0.37331143360651575     # locked value from model_meta.json

def metrics_at(y, s, thr):
    pred = (s >= thr).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(precision=p, recall=r, f1=f, tp=tp, fp=fp, fn=fn, tn=tn)

m  = metrics_at(y_val, scores, THRESHOLD)
roc = roc_auc_score(y_val, scores)
pr  = average_precision_score(y_val, scores)

print(f"threshold {THRESHOLD:.4f}")
print(f"  recall    {m['recall']:.3f}   ({m['tp']} of {m['tp']+m['fn']} failing lots caught)")
print(f"  precision {m['precision']:.3f}   ({m['tp']} of {m['tp']+m['fp']} flagged lots are real)")
print(f"  F1        {m['f1']:.3f}")
print(f"  ROC-AUC   {roc:.3f}      (0.5 = random)")
print(f"  PR-AUC    {pr:.3f}      (baseline = prevalence {y_val.mean():.3f})")

# Bootstrap. With 21 positives, a point estimate on its own is close to meaningless.
B = 2000
boot = {k: [] for k in ("recall", "precision", "roc", "pr")}
for _ in range(B):
    i = rng.integers(0, len(y_val), len(y_val))
    if y_val[i].sum() == 0 or y_val[i].sum() == len(i):
        continue
    mm = metrics_at(y_val[i], scores[i], THRESHOLD)
    boot["recall"].append(mm["recall"]); boot["precision"].append(mm["precision"])
    boot["roc"].append(roc_auc_score(y_val[i], scores[i]))
    boot["pr"].append(average_precision_score(y_val[i], scores[i]))

print(f"\n95% bootstrap CIs ({B} resamples):")
for k, label in [("recall","recall"), ("precision","precision"),
                 ("roc","ROC-AUC"), ("pr","PR-AUC")]:
    lo, hi = np.percentile(boot[k], [2.5, 97.5])
    print(f"  {label:<10} {np.mean(boot[k]):.3f}   [{lo:.3f}, {hi:.3f}]")
print("\nIf the ROC-AUC interval contains 0.5, the model is not distinguishable")
print("from random ranking on this validation set. Say so rather than quoting the mean.")

In [ ]:
#@title 5. Baselines — does it beat doing something trivial?
rows = []

def add(name, s):
    try:
        rows.append((name, roc_auc_score(y_val, s), average_precision_score(y_val, s)))
    except ValueError:
        pass

add("IsolationForest (shipped)", scores)
add("random scores", rng.random(len(y_val)))
add("max |z| across sensors", np.abs(Xva_v).max(axis=1))
add("mean |z| across sensors", np.abs(Xva_v).mean(axis=1))
add("count |z| > 3", (np.abs(Xva_v) > 3).sum(axis=1).astype(float))

# supervised reference — how much of the difficulty is the dataset, not the method?
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
lr = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xtr_v, y_train)
add("LogisticRegression (supervised)", lr.predict_proba(Xva_v)[:, 1])
hgb = HistGradientBoostingClassifier(random_state=RANDOM_STATE,
                                     class_weight="balanced").fit(Xtr_v, y_train)
add("HistGradientBoosting (supervised)", hgb.predict_proba(Xva_v)[:, 1])

print(f"{'method':<36}{'ROC-AUC':>10}{'PR-AUC':>10}")
print("-" * 56)
for name, r, p in sorted(rows, key=lambda t: -t[2]):
    print(f"{name:<36}{r:>10.3f}{p:>10.3f}")
print("-" * 56)
print(f"{'prevalence (PR-AUC floor)':<36}{0.5:>10.3f}{y_val.mean():>10.3f}")
print("\nPR-AUC is the metric that matters at 6.6% positives; ROC-AUC looks")
print("flattering on imbalanced data because true negatives are abundant and cheap.")

In [ ]:
#@title 6. Threshold sweep — the shipped 0.373 is one point on a curve
print(f"{'thr':>7}{'flagged':>9}{'recall':>9}{'prec':>8}{'F1':>8}   {'engineer workload':<28}")
print("-" * 72)
for thr in np.round(np.quantile(scores, np.linspace(0.50, 0.99, 12)), 4):
    mm = metrics_at(y_val, scores, thr)
    n_flag = mm["tp"] + mm["fp"]
    work = (f"review {n_flag:>3} lots -> find {mm['tp']:>2}" if n_flag else "flags nothing")
    mark = "  <-- shipped" if abs(thr - THRESHOLD) < 0.02 else ""
    print(f"{thr:>7.3f}{n_flag:>9}{mm['recall']:>9.3f}{mm['precision']:>8.3f}"
          f"{mm['f1']:>8.3f}   {work:<28}{mark}")

print("\nRead the last column, not the F1. The operating point is a staffing decision:")
print("how many lots an engineer will review to catch how many real failures.")

In [ ]:
#@title 7. Write eval_results.json
out = {
  "dataset": "SECOM (UCI id=179)",
  "n_val": int(len(y_val)), "n_fail_val": int(y_val.sum()),
  "threshold": THRESHOLD,
  "point_estimates": {k: (float(v) if not isinstance(v, (int, np.integer)) else int(v))
                      for k, v in m.items()} | {"roc_auc": float(roc), "pr_auc": float(pr)},
  "bootstrap_95ci": {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]
                     for k, v in boot.items()},
  "baselines": {n: {"roc_auc": float(r), "pr_auc": float(p)} for n, r, p in rows},
  "prevalence": float(y_val.mean()),
  "notes": [
    "Validation has only %d failing lots; point estimates carry very wide intervals."
      % int(y_val.sum()),
    "PR-AUC is the metric of record at this prevalence, not ROC-AUC or accuracy.",
    "contamination=0.05 is inert: it affects .predict() only, and no metric here uses it.",
  ],
}
with open("eval_results.json", "w") as f: json.dump(out, f, indent=2)
print(json.dumps(out["bootstrap_95ci"], indent=2))

try:
    from google.colab import files; files.download("eval_results.json")
except Exception:
    print("not on Colab — eval_results.json written locally")

## Back in the repo

```bash
mv ~/Downloads/eval_results.json src/models/tabular/eval_results.json
```

### How to read this

The vision model and this one are not the same kind of result, and the demo should not
present them as if they were. The classifier is genuinely good (macro-F1 0.92 on 9,357
maps). This detector is working on 21 failing lots in validation, on a dataset where
published work struggles to beat chance.

If the bootstrap interval for ROC-AUC spans 0.5, the honest statement is *"on SECOM we
cannot show this beats random ranking"* — and that is a better thing to say on stage than a
point estimate that a judge can dismantle with one question about sample size. It is also
the argument for the v2 direction: the value is in abstention and calibration, not in
squeezing another 0.02 out of an estimate whose error bar is ten times that wide.